## Topic 1: What a Vector DB Solves That SQL Can't

### Concept, Intuition, Why It Exists

- SQL databases are built around **exact match and structured filtering** — a query either matches a row or it doesn't, based on equality, ranges, or pattern matching. "Find me the FD record where FD_No = 'BJ2022FD6918'" is exactly what SQL is built for. It's deterministic, fast, and correct for structured lookups.
- The problem RAG introduces is a fundamentally different kind of question: "find me the chunks of text that are *most semantically similar* to this user query" — not equal to it, not containing a specific keyword, but *closest in meaning*. There's no SQL operator for that. `WHERE content LIKE '%premature withdrawal%'` would miss "early FD closure penalty" even though both phrases mean the same thing in this domain.
- **A vector database solves this by storing embeddings — fixed-size numerical vectors — alongside the original text, and building an index that lets you find the nearest vectors to a query vector efficiently.** Semantic closeness in the original text maps to geometric closeness in the vector space, and the vector DB's job is to find the geometrically closest stored vectors to a query vector as fast as possible.
- This is the natural next step after the chunking sub-chapter: each chunk now has a `page_content` string and a `metadata` dict — a vector DB adds a third thing, the embedding of that `page_content`, and builds an index over all those embeddings that supports "give me the top-k most similar stored chunks to this query."
- **Why a *database* and not just a list of vectors in memory?** This project's earlier chapters actually did use an in-memory list of vectors — stored as a Python list, searched by looping through every entry and computing similarity one by one. That works at the scale of a demo. It stops working the moment the corpus grows: it doesn't persist across restarts, doesn't support filtering by metadata, and the brute-force search loop scales linearly with corpus size. A vector DB solves all three at once.

### Internal Working

- **Storage**: a vector DB stores, for each indexed item: the embedding vector itself (a list of floats, typically 384–1536 dimensions depending on the model), the original text or a reference to it, and a metadata payload (structured fields like `source`, `page`, `chunk_index`, `document_code`).
- **Indexing**: instead of scanning every stored vector on every query (the in-memory brute-force approach), a vector DB builds a spatial index over all vectors that lets it find approximate nearest neighbors in sub-linear time. The specific index structure varies by system — covered in depth in the HNSW topic.
- **Query**: a user question gets embedded using the same model used at ingest time, producing a query vector. The DB searches its index for the stored vectors geometrically closest to that query vector (measured by cosine similarity or dot product for normalized vectors, or Euclidean distance), and returns the top-k matches along with their stored text and metadata.
- **Metadata filtering**: most production vector DBs support filtering the search space by metadata fields before or during the nearest-neighbor search — e.g. "only search among chunks where `source == '01_FD_FAQ.pdf'`" — combining semantic search with the kind of structured filtering SQL is good at.

### How It's Implemented In This Project

- Earlier chapters kept an in-memory list of `{"text":, "source":, "embedding":}` dicts, searched by iterating through every entry. This chapter migrates that exact structure into Qdrant — same three fields, now persisted in a real indexed collection, with metadata filtering available on top.
- The migration is designed to be minimal: the Document shape from the ingestion chapter maps directly onto Qdrant's data model with no structural redesign, deliberately showing that the abstraction choices made earlier were forward-compatible with a real vector DB without needing to be rebuilt from scratch.

### Real-World Issues, Edge Cases, Debugging

- **Approximate vs. exact nearest neighbors**: every practical vector index (HNSW, IVF) trades a small amount of recall for a large gain in search speed — the top-k results from an approximate search may occasionally miss a stored vector that is technically the closest, returning the second- or third-closest instead. For most RAG use cases this is acceptable; for applications requiring provably exact results, it's a real constraint.
- **Embedding model lock-in**: the moment vectors are stored in a vector DB, they're tied to the model that produced them. Every query must be embedded with that same model — if the model changes, every stored vector becomes incomparable to new queries. This is covered in full in the embedding model migration topic.
- **SQL and vector search are complementary, not mutually exclusive**: most production systems use both — SQL/structured storage for exact-match lookups (FD_No, customer records, status fields) and a vector DB for semantic search over unstructured content. The right mental model is "add a vector DB where semantic search is actually needed," not "replace all SQL with vector DB."

### Design Decisions & Trade-offs

- In-memory list vs. vector DB is not a quality trade-off — both produce the same search results at small scale. It's a scalability, persistence, and filtering trade-off: the in-memory approach is simpler, has zero infrastructure, and is perfectly valid for a demo corpus. The vector DB is the right choice once any of those three limitations actually matters — persistence across restarts, metadata-filtered search, or a corpus too large for brute-force scan to be fast enough.
- Using a vector DB doesn't mean abandoning exact-match lookup — `get_fd_record()` from Chapter 3 still uses the CSV for exact FD_No lookups, and that's still the right tool for that query pattern. Vector search is added for the queries that exact match can't handle, not as a replacement for the ones it handles well.

### Alternatives & When To Use Each

- In-memory brute-force (previous chapters) — corpus fits in RAM, no persistence needed, no metadata filtering required, simplicity prioritized.
- pgvector (Postgres extension) — team already uses Postgres and wants to avoid a new infrastructure component; accepts slightly lower performance than a dedicated vector DB.
- Dedicated vector DB (Qdrant, Weaviate, Pinecone) — corpus too large for brute-force, persistence required, metadata filtering needed, or query latency matters at scale.

### Common Mistakes & Production Failures

- Treating vector DB search as a drop-in SQL replacement for structured queries — "find me all FDs with status Active" is still a SQL/structured query, not a semantic search problem, and routing it through a vector DB is slower, less reliable, and more expensive than a direct filtered lookup.
- Using a different embedding model at query time than at ingest time — produces query vectors in a completely different geometric space from the stored vectors, guaranteeing that even the correct answer will rank near the bottom of the results.
- Not understanding that "approximate" in approximate nearest neighbor means occasional misses are expected behavior, not bugs — chasing a missing correct result through the application layer when the root cause is ANN recall is a common debugging dead end.

### Lead-Level Interview Questions

**Q: A junior engineer says "we can just use LIKE queries in Postgres to find relevant documents." What's the actual difference between that and vector search, and when does it matter?**
A: LIKE queries match literal substrings — they find documents containing the exact characters you searched for. Vector search finds documents with similar *meaning*, regardless of whether they share any words. The difference matters the moment users phrase questions differently from how answers are written in the source documents — which is almost always. "Early FD closure penalty" and "premature withdrawal charge" share no keywords but are semantically equivalent; LIKE would miss one while vector search finds both.

**Q: Your in-memory vector search is correct and fast enough today. What's the actual trigger to migrate to a vector DB, rather than migrating preemptively?**
A: Three concrete triggers: (1) the process restarts and the in-memory index is lost, forcing a full re-embed on startup — a real problem once re-embedding takes more than a few seconds; (2) corpus size makes per-query brute-force scan latency measurable and growing; (3) a query requirement needs metadata filtering that can't be done cleanly by filtering results after the fact. Before any of those three are real problems, migration is infrastructure cost with no user-visible benefit.

**Q: Should a vector DB fully replace the structured database used for exact FD record lookups in this project?**
A: No — they serve different query patterns. Exact-match lookups (by FD_No, by customer ID) are faster, cheaper, and more reliable in a structured store. Vector DB search is the right tool for semantic queries over unstructured content. The right architecture keeps both: structured DB for structured access patterns, vector DB for semantic search, with application logic routing each query to the appropriate system.

### Hidden Concepts Worth Knowing

- **Cosine similarity vs. dot product vs. Euclidean distance**: these are three different ways to measure vector closeness, and they give different results on un-normalized vectors. For normalized vectors (unit length), cosine similarity and dot product are equivalent — which is why the embedding steps throughout this project normalize vectors, making the cheaper dot product a valid substitute for the more expensive cosine similarity computation.
- **The curse of dimensionality**: in very high-dimensional spaces, distances between points tend to become less meaningful as a distinction — most points are "about equally far" from any query point. This is part of why embedding models target moderate dimensionality (384–1536) rather than higher — high enough to encode meaning, low enough that distance is still a discriminating signal.

### Revision Summary

> A vector DB stores embeddings alongside text and metadata, builds a spatial index for sub-linear nearest-neighbor search, and supports metadata filtering on top of semantic search — solving what SQL can't (semantic closeness), while SQL still handles what it was always better at (exact-match structured lookups). The in-memory approach from earlier chapters is correct at small scale and the right starting point; migration to a vector DB is triggered by persistence needs, corpus scale, or metadata filtering requirements — not by sophistication alone.

## Topic 2: Comparison — FAISS vs. Chroma vs. Qdrant vs. Pinecone vs. Weaviate

### Concept, Intuition, Why It Exists

- Not all "vector search" tools are the same kind of thing — some are libraries, some are embedded databases, some are fully managed cloud services. Knowing the distinction matters because the right choice depends on what stage of a project you're in, what infrastructure you already have, and what operational requirements actually exist.
- **FAISS** (Facebook AI Similarity Search): a C++ library with a Python wrapper, not a database at all. It provides highly optimized index structures (Flat, IVF, HNSW, PQ) for nearest-neighbor search over vectors stored in memory or on disk. No server, no persistence layer, no metadata filtering, no REST API — just raw index-and-search over numpy arrays. The fastest option at pure search speed for a given index type, but you're responsible for everything else: serializing the index, storing the original text, handling metadata, managing persistence across restarts.
- **Chroma**: an embedded vector database (runs in-process, no separate server needed) designed specifically for rapid prototyping in the LangChain/LlamaIndex ecosystem. Extremely easy to get started — a few lines of Python, no Docker, no cloud account. The trade-off is that it's embedded-first, meaning running it at scale across multiple processes or machines requires stepping outside its simplest use mode. Good for notebooks and small projects, less suitable as a production-grade persistent store.
- **Qdrant**: an open-source, dedicated vector database written in Rust. Runs as a standalone server (Docker locally, or Qdrant Cloud's managed tier). Supports HNSW indexing, rich payload (metadata) filtering during search, both REST and gRPC APIs, on-disk storage, collections, and point-level access control. Genuinely production-ready without needing a paid tier — the local Docker image is the same codebase as the cloud offering. This project's choice, covered in full in the next topic.
- **Pinecone**: a fully managed, cloud-only vector database — no self-hosting option at all. Simplest operational overhead of any option here (no infrastructure to manage), and scales transparently. Trade-offs: vendor lock-in (your vectors live on Pinecone's servers, with no local option), cost at scale (paid tiers), and less control over indexing behavior than a self-hosted system.
- **Weaviate**: an open-source vector database with a broader scope than Qdrant — it has its own built-in vectorization (can call an embedding model on your behalf at ingest time), a graph-style object schema, and a GraphQL query API alongside REST. More feature-rich than Qdrant, but also heavier to configure and run — the additional features come with real setup complexity overhead.

### Internal Working

- **FAISS**: you manage everything manually — build an index object, add numpy arrays of vectors to it, call `index.search(query_vector, k)` to get back distances and integer IDs, then look up the original texts yourself by mapping those IDs back to your own storage. Serialization/deserialization to disk is your responsibility via `faiss.write_index` / `faiss.read_index`.
- **Chroma**: wraps the storage and retrieval loop — you pass text and metadata at add time (Chroma calls the embedding model you configure), then call `collection.query(query_texts=[...], n_results=k)` and get back documents, metadata, and distances in one call. The embedding, storage, and ID management are handled internally.
- **Qdrant**: you embed text yourself (Qdrant doesn't call embedding models on your behalf by default), then upsert points (id + vector + payload) into a named collection via REST or gRPC. At query time, you embed the query yourself and call the search endpoint with the query vector, optional payload filter, and k. Qdrant handles HNSW indexing, persistence, and filtering internally.
- **Pinecone**: identical interaction model to Qdrant from the client's perspective — you embed yourself, upsert vectors with metadata, query with an embedded vector and receive top-k matches — but everything runs on Pinecone's managed infrastructure rather than a server you run.
- **Weaviate**: defines a schema upfront (class definitions with typed properties), then adds objects that can be auto-vectorized if a vectorizer module is configured. Queries use either a GraphQL interface or a REST endpoint with `nearVector` or `nearText` operators.

### How It's Implemented In This Project

- This project's earlier in-memory approach was closest to what FAISS provides in its simplest form — raw vectors in a list, brute-force search loop, no persistence.
- Chroma would have been the easiest step up from that (embedded, minimal setup), but was deliberately not chosen — the project's requirements (real metadata filtering, Docker-or-cloud flexibility, a path to production without swapping tools) made Qdrant the better fit, even at the cost of needing a running server.
- FAISS is not used directly here, but understanding it is important because many other tools (including Qdrant's own index) are built on similar principles to FAISS's HNSW implementation.

### Real-World Issues, Edge Cases, Debugging

- **FAISS's biggest practical pain point is the ID management problem**: FAISS only knows about integer IDs, not text or metadata — you're responsible for maintaining your own mapping from FAISS integer IDs back to the original content. In practice this usually means a separate dict or database table, and keeping that mapping in sync with the FAISS index is a real operational burden as documents are updated or deleted.
- **Chroma's embedded mode hits limits when multiple processes need to write simultaneously**: since it runs in-process, two separate ingestion workers pointing at the same Chroma store can corrupt it. Chroma does have a client/server mode that solves this, but that's no longer simpler to operate than just running Qdrant directly.
- **Pinecone's vendor lock-in is real and asymmetric**: migrating *into* Pinecone is easy (their client library handles it). Migrating *out* means re-embedding your entire corpus from scratch into a new system, since Pinecone doesn't export raw stored vectors in a portable format — worth knowing before committing a large corpus to it.
- **Weaviate's auto-vectorization is convenient but creates a hidden dependency**: if Weaviate is configured to call an external embedding API on your behalf, your ingest pipeline now has a network call to an external service baked in at ingest time rather than at your own controlled embedding step. A rate limit or outage in that external service blocks ingestion, not a separate, visible embedding step you can retry independently.

### Design Decisions & Trade-offs

- FAISS — choose when maximum raw search performance matters and you're willing to own all surrounding infrastructure (persistence, metadata, ID mapping) yourself. Common in research and performance-critical systems with existing infrastructure.
- Chroma — choose when getting something working quickly in a notebook or demo matters more than production readiness. A genuinely good choice for early prototyping; honest about not being the right long-term tool for high-scale or multi-process setups.
- Qdrant — choose when you need production-grade persistence, real metadata filtering, and local-or-cloud flexibility without vendor lock-in and without the overhead of Weaviate's schema/GraphQL layer. This project's choice.
- Pinecone — choose when zero infrastructure management matters more than cost or vendor independence. A reasonable choice for small-to-medium corpora where the managed tier pricing is acceptable.
- Weaviate — choose when you want built-in auto-vectorization, a richer object schema, and are willing to invest in the configuration complexity that comes with it. More common in enterprise settings with dedicated ML infrastructure teams.

### Common Mistakes & Production Failures

- Choosing Chroma for production purely because it was the simplest to use in the initial prototype, without verifying it handles the actual production requirements (concurrent writes, large corpus, persistence guarantees).
- Choosing Pinecone without explicitly pricing the corpus size at scale — the cost curve is non-linear and can become significant at hundreds of millions of vectors.
- Using FAISS without a robust ID-to-content mapping layer and discovering after a re-index that the integer IDs no longer map to the same content they did before, because FAISS assigns IDs sequentially and doesn't handle deletion gracefully.
- Treating all these tools as interchangeable at the code level — switching from Chroma to Qdrant in an existing project is not just a config change, it involves adopting a different data model (collections/points/payloads vs. Chroma's collections/documents/metadata), different client libraries, and different query semantics.

### Lead-Level Interview Questions

**Q: Your team is three engineers, the corpus is 50,000 chunks, and the project is three months old. Someone proposes Pinecone. What's your actual evaluation?**
A: At 50,000 chunks Pinecone's free tier or starter tier likely covers it, so cost isn't immediately the issue. The real questions are: (1) does this team want to own zero infrastructure, or is running a Docker container acceptable? (2) how likely is a corpus migration to a different system in the next year — if likely, Pinecone's lack of vector export is a meaningful lock-in risk. (3) is metadata filtering needed during search? If yes, Pinecone supports it, but so does Qdrant with more flexibility. There's no universally wrong answer, but the trade-offs should be stated explicitly rather than defaulting to Pinecone because it's the easiest to start.

**Q: FAISS is faster than Qdrant at raw search. When would you still choose Qdrant?**
A: FAISS's speed advantage only holds for pure vector search on vectors that fit in memory, with no metadata filtering, no persistence across restarts, and no multi-client access. The moment you need any of those — and most production systems need at least one — the gap between FAISS's raw search speed and Qdrant's slightly slower but persistent, filterable, multi-client-safe search narrows to irrelevance compared to the operational complexity FAISS leaves entirely to you.

**Q: A teammate says "Weaviate does vectorization for us, which saves an entire step in the pipeline." Is that a benefit or a risk?**
A: Both, and the answer depends on what you value. The benefit is real — ingest pipeline is simpler if you don't manage an embedding step yourself. The risk is also real — you've now created an implicit dependency on Weaviate's configured vectorizer module being available, correctly configured, and using the exact same model version at ingest time and query time. Any deviation (a module update changing model behavior, a rate limit on an external vectorizer API) becomes a silent quality regression or an outage at ingest time. Owning your embedding step explicitly is less convenient but far more debuggable.

### Hidden Concepts Worth Knowing

- **Embedded vs. client-server architecture**: Chroma (embedded mode) runs inside your Python process — simple but means every process that needs the index must load it into its own memory. Qdrant, Pinecone, and Weaviate are all client-server — the index lives in a separate process or service, and multiple clients can query it concurrently without each needing a full copy of the index in memory. This distinction matters as soon as more than one thing needs to read or write the index at the same time.
- **gRPC vs. REST**: Qdrant supports both. REST is easier to debug (human-readable, curl-testable). gRPC is faster (binary protocol, lower serialization overhead) and worth adopting for high-throughput ingest pipelines. Starting with REST and switching to gRPC when throughput actually matters is the sensible progression.

### Revision Summary

> FAISS is a fast search library, not a database — fast but requires you to own persistence, ID mapping, and metadata entirely. Chroma is an embedded DB, great for prototyping, less suited for production concurrent workloads. Qdrant is open-source, production-ready, locally runnable, and this project's choice. Pinecone is fully managed with zero infrastructure overhead but real vendor lock-in and no vector export. Weaviate adds auto-vectorization and a richer schema at the cost of more configuration complexity. Choose based on actual requirements — persistence, filtering, scale, infrastructure budget — not on which is most sophisticated.

## Topic 3: Why Qdrant Specifically

### Concept, Intuition, Why It Exists

- Not all "vector search" tools are the same kind of thing — some are libraries, some are embedded databases, some are fully managed cloud services. Knowing the distinction matters because the right choice depends on what stage of a project you're in, what infrastructure you already have, and what operational requirements actually exist.
- The requirements this project actually has: runs locally without needing a cloud account or a paid tier, persists across restarts, supports metadata filtering during search, is production-ready enough that switching to a hosted version later doesn't require rewriting client code, and is open-source enough that the full behavior is inspectable rather than a black box managed service.
- Qdrant meets all five. That's the answer — not "Qdrant is the best vector DB", which isn't meaningfully true across all contexts, but "Qdrant is the right fit for these specific requirements at this scale."
- **For this project specifically**: Qdrant's Python client supports an in-memory mode (`QdrantClient(":memory:")`) that needs no Docker, no server, no network download — all the same concepts (collections, points, payloads, filtering, search) work identically. This is what this project uses for learning. The switch to a real persistent server is a one-line URL change when actually needed.

### Internal Working

- **Rust-based core**: written in Rust, giving memory safety without garbage collection pauses — relevant for latency-sensitive search paths where GC pauses in a Java or Python-based system cause occasional slow outlier queries.
- **HNSW indexing by default**: every collection uses Hierarchical Navigable Small World indexing (covered in the next topic). Fast approximate nearest-neighbor search with configurable parameters.
- **Payload filtering**: Qdrant allows filtering on stored payload fields as part of the vector search itself — not as a post-search filter applied after retrieving results, but as a constraint applied during the search, so the k returned results are already filtered. This is architecturally distinct from FAISS, where filtering is always a post-search step.
- **Collections and points**: the primary data model. A collection is a named group of points; each point has an ID, a vector, and a payload (a dict of arbitrary JSON-serializable fields). Covered in full in Topic 5.
- **REST and gRPC**: both APIs expose the same functionality. REST is the default starting point — easier to debug, curl-testable. gRPC is available when throughput matters at scale.
- **Three modes, same client code**:
  - `:memory:` — in-process, no server, no persistence, zero setup. Used in this project for learning.
  - Local Docker — persistent, same codebase as cloud, runs on your machine.
  - Qdrant Cloud — managed hosted version, one-line URL change from local Docker.

### How It's Implemented In This Project

- This project uses `QdrantClient(":memory:")` — no Docker required, no network download, starts instantly.
- All code in this chapter works identically in memory mode and Docker mode. The only thing that changes between them is one line in `get_client()`.
- The collection created (`fd_knowledge_base`) maps directly onto the Document shape from the ingestion chapter: `page_content` becomes text stored in the payload, `metadata` fields become payload fields, and the embedding of `page_content` becomes the point's vector.
- When you eventually want persistence, the upgrade path is: run `docker pull qdrant/qdrant`, start the container, change `QdrantClient(":memory:")` to `QdrantClient(url="http://localhost:6333")`. Nothing else changes.

### Real-World Issues, Edge Cases, Debugging

- **In-memory mode loses all data on restart**: this is the one real limitation of `:memory:` mode — every notebook restart means re-running the upsert cells. Acceptable for learning, not acceptable for production.
- **Docker volume mount is critical for persistence**: when switching to Docker later, running without `-v` loses all data on container restart. Always use: `docker run -p 6333:6333 -v <path>/qdrant_storage:/qdrant/storage qdrant/qdrant`.
- **Collection already exists**: if you run the collection creation code twice, Qdrant raises an error — not a warning. Production code needs to check for existence before creating, or explicitly use `recreate=True`.
- **Vector dimension mismatch**: a collection is created with a fixed vector size (384 for `paraphrase-multilingual-MiniLM-L12-v2`). Trying to upsert a point with a different-dimensioned vector raises an error immediately. The most common issue during an embedding model swap.
- **`:memory:` is not the same as Chroma's embedded mode**: Qdrant's in-memory client is a fully-featured Qdrant instance running in-process — payload filtering, HNSW indexing, all APIs work. It is not a stripped-down development toy.

### Design Decisions & Trade-offs

- In-memory for learning, Docker for persistence, Cloud for production — three modes, one codebase, zero rewriting. This is the specific property that makes Qdrant the right choice here: the upgrade path between modes is a one-line change, not a migration.
- gRPC deferred: REST is used because it is easier to debug at learning stage. gRPC is available and worth knowing exists for production high-throughput scenarios.
- Qdrant over Chroma: Chroma's embedded mode also needs no Docker, but doesn't support payload filtering during search and has concurrent-writer limitations. Qdrant's `:memory:` mode gives the same zero-setup experience with full production-equivalent behavior.

### Alternatives & When To Use Each

- `QdrantClient(":memory:")` — learning, notebooks, CI tests, any context where persistence isn't needed and zero setup matters.
- Local Docker — development and production on your own machine, persistence required, same code as cloud.
- Qdrant Cloud — managed hosting, zero infrastructure, one-line change from local Docker, no vendor lock-in on vector format unlike Pinecone.
- Chroma embedded — if Qdrant's client isn't installable for some reason and metadata filtering isn't needed.

### Common Mistakes & Production Failures

- Using `:memory:` mode in production and wondering why all vectors disappear on every restart.
- Switching to Docker later without the `-v` volume mount and silently losing all indexed data on the first container restart.
- Creating a collection with one embedding model's vector size, then swapping the embedding model without recreating the collection — Qdrant rejects every upsert with a dimension mismatch error.
- Not knowing the dashboard exists: when running Docker, `http://localhost:6333/dashboard` lets you inspect collections, run test queries, and verify upserts worked without writing diagnostic code.

### Lead-Level Interview Questions

**Q: Your team can't get Docker running in a restricted environment. Does that block using Qdrant?**
A: No — `QdrantClient(":memory:")` runs entirely in-process with no server, no Docker, no network. All the same APIs, filtering, and indexing behavior works. The only thing lost is persistence across restarts, which is a real limitation for production but irrelevant for development, testing, and learning.

**Q: What is the actual upgrade path from in-memory Qdrant to a persistent production deployment?**
A: Change one line: `QdrantClient(":memory:")` becomes `QdrantClient(url="http://localhost:6333")` for local Docker, or `QdrantClient(url="https://your-cluster.qdrant.io", api_key="...")` for Qdrant Cloud. Every other line of code — collection creation, upsert, search, filtering — stays identical. This is the specific property that makes `:memory:` mode genuinely useful rather than a throwaway prototype.

**Q: What does "payload filtering during search" mean in Qdrant, and why does it matter compared to filtering results after the fact?**
A: In Qdrant, a payload filter is applied as a constraint inside the HNSW search itself — the index only considers points matching the filter during traversal, so the k results returned already satisfy the filter. Post-search filtering means fetching a larger result set and discarding non-matching entries afterward, which either wastes the overfetch or returns fewer than k useful results if you don't overfetch enough.

**Q: Why use Qdrant over Chroma given both can run without Docker?**
A: Chroma's embedded mode removes the server step, which is genuinely convenient. The trade-offs: Chroma doesn't support payload filtering during search (only post-search), has concurrent-writer limitations in embedded mode, and requires a migration when the project outgrows it. Qdrant's `:memory:` mode gives the same zero-setup experience with full production-equivalent filtering behavior and a one-line upgrade path to a real server.

### Hidden Concepts Worth Knowing

- **Collection aliases**: Qdrant supports creating a named alias pointing to a collection. This enables the build-then-swap pattern — ingest into a staging collection, validate, then atomically update the alias to point at the new collection. Live queries always hit the alias, so the swap is instantaneous and no query lands in a half-updated state.
- **Sparse vectors and hybrid search**: Qdrant supports sparse vectors (BM25-style keyword representations) alongside dense vectors, enabling hybrid search — a combination of semantic similarity and keyword relevance in one query. Relevant when semantic search alone misses exact-match queries like specific FD reference numbers.

### Revision Summary

> Qdrant is chosen because it meets all five concrete requirements: zero-setup in-memory mode for learning, Docker for local persistence, Qdrant Cloud for managed hosting — all with identical client code and a one-line upgrade path between modes. `QdrantClient(":memory:")` is what this project uses: no Docker needed, full Qdrant behavior, data lost on restart. When persistence is actually needed, one line changes. Nothing else does.

In [2]:
"""
qdrant_setup.py
-----------------
Qdrant setup using in-memory mode -- no Docker, no server, no network
download required. All Qdrant behavior (collections, points, payloads,
filtering, search) works identically in :memory: mode and Docker mode.

The ONLY difference between modes is one line in get_client():
  - Learning / no Docker : QdrantClient(":memory:")
  - Local Docker         : QdrantClient(url="http://localhost:6333")
  - Qdrant Cloud         : QdrantClient(url="https://...", api_key="...")

When you want persistence later:
    docker run -p 6333:6333 -v <your_path>/qdrant_storage:/qdrant/storage qdrant/qdrant
    Then swap get_client() to QdrantClient(url="http://localhost:6333").

Install: pip install qdrant-client
"""

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

COLLECTION_NAME = "fd_knowledge_base"
VECTOR_SIZE = 384        # paraphrase-multilingual-MiniLM-L12-v2 output dim


def get_client() -> QdrantClient:
    """
    Returns a connected Qdrant client.

    Currently uses :memory: mode -- no Docker needed, zero setup,
    data is lost when the Python process restarts.

    To switch to persistent local Docker, replace with:
        return QdrantClient(url="http://localhost:6333")

    To switch to Qdrant Cloud, replace with:
        return QdrantClient(url="https://your-cluster.qdrant.io", api_key="YOUR_KEY")
    """
    return QdrantClient(":memory:")


def create_collection(client: QdrantClient, recreate: bool = False) -> None:
    """
    Creates the collection if it doesn't already exist.

    recreate=True drops and rebuilds it -- use this when changing embedding
    models or chunk schemas, never on a live collection with real data.
    """
    existing = [c.name for c in client.get_collections().collections]

    if COLLECTION_NAME in existing:
        if recreate:
            client.delete_collection(COLLECTION_NAME)
            print(f"Deleted existing collection '{COLLECTION_NAME}'")
        else:
            print(f"Collection '{COLLECTION_NAME}' already exists -- skipping creation")
            return

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,
            distance=Distance.COSINE,
        ),
    )
    print(f"Created collection '{COLLECTION_NAME}' (vector size={VECTOR_SIZE})")


def verify_connection(client: QdrantClient) -> None:
    """Quick sanity check -- prints collection info if everything is working."""
    info = client.get_collection(COLLECTION_NAME)
    print(f"Collection   : {COLLECTION_NAME}")
    print(f"Vector size  : {info.config.params.vectors.size}")
    print(f"Points count : {info.points_count}")
    print(f"Status       : {info.status}")


if __name__ == "__main__":
    client = get_client()
    create_collection(client, recreate=False)
    verify_connection(client)

Created collection 'fd_knowledge_base' (vector size=384)
Collection   : fd_knowledge_base
Vector size  : 384
Points count : 0
Status       : green


## Topic 4: Indexing & HNSW Explained

### Concept, Intuition, Why It Exists

**What is an index, and why does it exist at all?**

- When you store vectors in a flat list and search it, you are doing **brute-force search** — computing the similarity between your query vector and every single stored vector, one by one, then sorting to find the top-k. This is called an **exact nearest neighbor search** — it is guaranteed to return the true closest vectors, no approximations.
- The problem is that brute-force search scales linearly: double the number of stored vectors, double the search time. At 1,000 vectors this is invisible. At 1,000,000 vectors this is seconds per query. At 100,000,000 vectors this is completely infeasible.
- An **index** is a data structure built over your vectors, before any queries arrive, that reorganizes them in a way that lets you skip most of the comparisons during search. Instead of checking every vector, the index lets you check only a small, intelligently selected subset — vectors that are likely to be close to the query — and still return results that are almost always correct.
- The trade-off is fundamental and explicit: **exact search** (check everything, guaranteed correct, slow at scale) vs. **approximate nearest neighbor (ANN) search** (check a smart subset, occasionally misses the single closest vector, fast at any scale). Almost every production vector search system accepts this trade-off — the occasional miss is worth the orders-of-magnitude speed gain.
- This project's earlier in-memory list of vectors used brute-force exact search. Moving to Qdrant means moving to indexed ANN search — the same conceptual operation, done in a fundamentally different and scalable way.

**What is HNSW?**

- **Hierarchical Navigable Small World** is the index structure Qdrant uses by default, and the most widely used ANN index in production vector databases (also used internally by FAISS, Weaviate, and others).
- The name describes exactly what it is: a **graph** of vectors, organized in **multiple layers** (hierarchical), where each node is connected to a small number of nearby neighbors (navigable small world), and search means traversing that graph to find the closest nodes to a query — rather than scanning a flat list.

### Internal Working

**How HNSW is built (the index construction phase):**

1. Vectors are inserted one at a time into the graph. Each vector becomes a **node**.
2. Each new node is assigned a **maximum layer** by a random process — most nodes exist only in layer 0 (the bottom, densest layer), fewer in layer 1, fewer still in layer 2, and so on. This exponentially thinning distribution across layers is what makes the structure hierarchical.
3. In each layer the node exists in, it is connected to its `m` nearest neighbors **at that layer** — `m` is a key parameter (typically 16–64) that controls the graph's density. More connections = better recall, more memory.
4. The result is a multi-layer graph where the top layers have few nodes and long-range connections (good for quickly navigating to the right general region of the space), and the bottom layer (layer 0) has all nodes and short-range connections (good for precise local search once you're in the right region).

**How HNSW search works (the query phase):**

1. Start at the entry point — a fixed node in the topmost layer.
2. **Greedy traversal downward**: at each layer, follow edges to whichever neighbor is closest to the query vector, moving greedily toward the query until no neighbor is closer than the current node. This is a local greedy search — fast, but approximate.
3. Drop down to the next layer, using the best node found in the layer above as the new starting point. Repeat.
4. At layer 0 (the densest layer with all nodes), do a more thorough local search using a parameter called `ef` (the search-time exploration factor) — explore `ef` candidate neighbors instead of just greedily following the single best at each step. Higher `ef` = better recall, slower search.
5. Return the top-k nodes from the layer 0 search as the approximate nearest neighbors.

**Why the hierarchical structure helps:**

- Top layers act like a coarse map — they let search jump quickly to the right region of the space without examining every node.
- Bottom layers act like a detailed local search — once you're in the right region, you search carefully among nearby nodes.
- This is exactly analogous to how you'd search a city: first navigate to the right neighborhood (top layers), then look carefully for the specific address (bottom layer). Checking every address in the city from the start would be brute-force; using the neighborhood structure is HNSW.

**Key parameters:**

- `m`: number of edges per node per layer. Higher = better recall, more memory, slower build. Default 16, production systems often use 32–64.
- `ef_construction`: how thoroughly to search for neighbors during index build. Higher = better index quality, slower build time. Default 100.
- `ef` (search time): how thoroughly to search at query time. Higher = better recall, slower query. Can be tuned per-query without rebuilding the index.

### How It's Implemented In This Project

- Qdrant uses HNSW by default for every collection. When `create_collection()` is called with `VectorParams(size=384, distance=Distance.COSINE)`, Qdrant automatically builds and maintains an HNSW index over all upserted vectors — no explicit index construction step needed, it happens incrementally as points are added.
- The default `m` and `ef_construction` values Qdrant uses are sensible for most corpora. At this project's scale (hundreds of chunks, not millions), the index parameters make no measurable difference — they matter when the corpus grows into the tens of thousands of vectors.
- Qdrant's payload filtering during search (used in the metadata filtering and PII topics later) is implemented by constraining which nodes the HNSW traversal is allowed to visit — only nodes whose payload matches the filter are valid candidates. This is more efficient than post-search filtering because the traversal never wastes time on non-matching nodes.

### Real-World Issues, Edge Cases, Debugging

- **ANN recall is not 100%**: HNSW will occasionally miss the single true nearest neighbor, returning the second or third closest instead. For RAG, this is almost always acceptable — the difference between the first and second closest chunk is rarely the difference between a correct and incorrect answer. For applications requiring provably exact results (fraud detection thresholds, legal document matching), ANN is not appropriate without additional exact-verification steps.
- **Index build time vs. query time trade-off**: high `ef_construction` and `m` produce a better quality index but take longer to build and use more memory. For a batch ingest of a large corpus, this matters. For this project's scale, it doesn't.
- **Filtering and recall interact**: when payload filtering is applied during HNSW search, the effective search space shrinks. If the filter is very restrictive (e.g. "only search 10 points out of 10,000"), HNSW traversal may struggle to find enough valid candidates and return fewer than k results. Qdrant handles this gracefully but it's worth knowing that aggressive filtering on small subsets can reduce effective recall.
- **HNSW is not updateable in place at low cost**: deleting a node from an HNSW graph doesn't cleanly remove it — Qdrant marks it as deleted and excludes it from results, but the graph edges remain, slightly degrading traversal efficiency over time. Periodic re-indexing (rebuilding the collection) recovers that efficiency for large-scale systems with many deletions.

### Design Decisions & Trade-offs

- HNSW vs. flat (brute-force exact search): flat search is actually the right choice at very small scale (hundreds of vectors) because there's no index build overhead and exact results are guaranteed. HNSW becomes the right choice as the corpus grows, accepting occasional misses in exchange for sub-linear search time. Qdrant uses HNSW by default regardless of collection size, which means at very small scales you're paying a small build overhead for an index you don't strictly need — a sensible default for a system that expects to grow.
- HNSW vs. IVF (Inverted File Index, used by FAISS): IVF partitions vectors into clusters (using k-means) and at search time only examines the clusters closest to the query. IVF is faster to build than HNSW and uses less memory, but has worse recall at the same search budget. HNSW has better recall for the same query time, at the cost of more memory. For most RAG workloads, recall matters more than memory, so HNSW is the better default.

### Alternatives & When To Use Each

- Flat / brute-force exact search — corpus fits in memory, exact results required, scale is small enough that linear scan is fast. Qdrant supports this explicitly with `VectorParams(on_disk=False)` and no HNSW — useful for tiny reference collections.
- HNSW (Qdrant default) — general-purpose production default, excellent recall, scales well, higher memory than IVF.
- IVF (FAISS) — very large corpora where memory is the primary constraint and slightly lower recall is acceptable.
- ScaNN, DiskANN — specialized systems for billion-scale corpora or on-disk indexing where even HNSW's memory requirements become prohibitive.

### Common Mistakes & Production Failures

- Treating ANN recall as binary ("it either works or it doesn't") rather than as a tunable parameter — increasing `ef` at query time directly improves recall at the cost of slightly slower search, and this trade-off can be adjusted without rebuilding the index.
- Expecting HNSW to handle very aggressive metadata filtering well on tiny subsets — if the filter leaves only 5 valid points in a 50,000-point collection, HNSW will struggle to find them efficiently, and a filtered brute-force scan of those 5 points would actually be faster and more correct.
- Not accounting for HNSW memory usage when sizing infrastructure — an HNSW index for 10 million 384-dimensional vectors can consume several GB of RAM beyond the raw vector storage, depending on `m`.

### Lead-Level Interview Questions

**Q: Why is approximate nearest neighbor search acceptable for RAG, but might not be acceptable for other applications?**
A: In RAG, the goal is to retrieve chunks that are relevant enough for the LLM to produce a correct answer. The difference between the first and second closest chunk is rarely meaningful — both are likely relevant. In contrast, applications like fraud detection (is this transaction within a threshold distance of known fraud patterns?) or biometric matching (is this face embedding the closest match to a stored identity?) may require provably exact nearest neighbors, because the specific closest result, not just a close one, is the decision boundary.

**Q: HNSW occasionally misses the true nearest neighbor. How would you detect whether this is actually impacting retrieval quality in production?**
A: Build a small evaluation set of queries with known correct chunks (the chunks that should be retrieved for each query, verified by hand or by exact search). Run production HNSW search and measure recall at k — the fraction of queries where the known correct chunk appears in the top-k results. If recall is below acceptable, increase `ef` at search time, which improves recall without rebuilding the index. If that's insufficient, consider increasing `m` and rebuilding.

**Q: A collection has 10,000 points but a metadata filter restricts search to 20 of them. HNSW returns fewer than the requested k results. Why, and what's the fix?**
A: HNSW traversal is guided by the graph structure built over all 10,000 points — it navigates toward the query through the full graph, but can only collect results from the 20 matching points. If the graph's traversal path doesn't naturally pass through enough of those 20 points within the `ef` search budget, it returns fewer than k. The fix is either increasing `ef` specifically for heavily-filtered queries, or — for very small filtered subsets — switching to a brute-force scan of just the matching points, which Qdrant can do automatically when the filtered subset is small enough.

**Q: What's the difference between `ef_construction` and `ef`, and which one would you tune first in production?**
A: `ef_construction` is fixed at index build time — it controls how thoroughly neighbors are searched when building the graph, affecting index quality permanently. `ef` is a per-query parameter — it controls how thoroughly the graph is traversed at search time, affecting recall and latency for each query without touching the index. Tune `ef` first in production, because it's a zero-cost change (no rebuild, takes effect immediately) that directly improves recall. Only increase `ef_construction` and rebuild the index if the index quality itself is the bottleneck — which requires a controlled experiment comparing recall at high `ef` against an index rebuilt with higher `ef_construction`.

### Hidden Concepts Worth Knowing

- **Small world graphs**: the "small world" property (from network science) means that in a well-connected graph, any two nodes can be reached from each other in a small number of hops — like the "six degrees of separation" idea. HNSW deliberately constructs a graph with this property so that traversal from any starting point to any target can complete in logarithmic hops rather than linear ones.
- **The curse of dimensionality and why it doesn't kill HNSW**: in very high dimensions, distances between random points converge — most points are "about equally far" from any query. HNSW partially sidesteps this because it builds connections based on relative proximity during construction, not random sampling — the graph structure captures local neighborhood relationships that remain meaningful even in high dimensions.
- **HNSW memory layout**: the graph edges (which node connects to which) are stored separately from the vectors themselves. Qdrant can store vectors on disk while keeping the graph structure in memory, enabling larger-than-RAM corpora at the cost of disk I/O for vector reads during search — controlled by `on_disk=True` in the collection config.

### Revision Summary

> An index exists to make nearest-neighbor search sub-linear — instead of comparing a query vector to every stored vector, an index lets search skip most comparisons by navigating a pre-built structure. HNSW builds a multi-layer graph where top layers enable fast coarse navigation to the right region of the vector space, and the bottom layer enables precise local search once there. The trade-off is exact vs. approximate: HNSW occasionally misses the single true nearest neighbor, which is almost always acceptable for RAG in exchange for orders-of-magnitude faster search. Key tuning knobs: `m` (graph density, set at build time), `ef_construction` (build quality, set at build time), `ef` (search recall, tunable per query without rebuilding).

In [5]:
"""
qdrant_hnsw.py
----------------
Demonstrates HNSW index parameter control in Qdrant.
Uses :memory: mode -- no Docker required.

Install: pip install qdrant-client sentence-transformers
"""

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,       # tells Qdrant how to measure closeness between vectors
    VectorParams,   # defines the vector size and distance metric for a collection
    HnswConfigDiff, # lets us override the default HNSW index parameters
    PointStruct,    # the object that holds one point: id + vector + payload
    Filter,         # wraps one or more conditions for a filtered search
    FieldCondition, # one condition: "this payload field must match this value"
    MatchValue,     # the "must equal this exact value" part of a FieldCondition
    SearchParams,   # lets us control ef (search-time recall vs. speed trade-off)
)
from sentence_transformers import SentenceTransformer

COLLECTION_NAME = "fd_hnsw_demo"
VECTOR_SIZE = 384       # must match the embedding model's output dimension
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

# :memory: means no Docker, no server -- everything lives in this Python process
client = QdrantClient(":memory:")

# load the embedding model once -- reused for every encode() call below
model = SentenceTransformer(MODEL_NAME)

# small set of sample chunks -- in a real pipeline these come from chunker.py
CHUNKS = [
    {"text": "Premature withdrawal incurs a 1 percent penalty on the applicable rate.", "source": "FD_Policy", "doc_type": "policy"},
    {"text": "This penalty does not apply if the FD is closed due to the death of the depositor.", "source": "FD_Policy", "doc_type": "policy"},
    {"text": "Senior citizens receive an additional 0.5 percent interest on all tenures.", "source": "FD_Product_Guide", "doc_type": "product"},
    {"text": "What is the minimum amount to open an FD? The minimum deposit is Rs 25,000.", "source": "FD_FAQ", "doc_type": "faq"},
    {"text": "Can I withdraw my FD before maturity? Yes, subject to a penalty.", "source": "FD_FAQ", "doc_type": "faq"},
    {"text": "The FD interest rate for 24 months is 7.1 percent per annum.", "source": "FD_Product_Guide", "doc_type": "product"},
    {"text": "Nomination facility is available for all Fixed Deposit accounts.", "source": "FD_Policy", "doc_type": "policy"},
    {"text": "TDS is deducted at source if interest income exceeds Rs 40,000 per year.", "source": "FD_Policy", "doc_type": "policy"},
]


def create_hnsw_collection(m: int = 16, ef_construction: int = 100) -> None:
    # check what collections already exist so we don't crash on a duplicate name
    existing = [c.name for c in client.get_collections().collections]

    # delete the old collection if it exists -- clean slate for this demo
    if COLLECTION_NAME in existing:
        client.delete_collection(COLLECTION_NAME)

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,       # every vector in this collection must be 384 floats
            distance=Distance.COSINE,  # cosine similarity -- right choice for normalized vectors
        ),
        hnsw_config=HnswConfigDiff(
            m=m,                    # edges per node -- higher = better recall, more memory
            ef_construct=ef_construction,  # build-time search depth -- higher = better index quality
        ),
    )
    print(f"Created collection with HNSW m={m}, ef_construction={ef_construction}")


def upsert_chunks() -> None:
    # extract just the text strings so we can embed them all in one batch call
    texts = [c["text"] for c in CHUNKS]

    # embed all texts at once -- normalize_embeddings=True makes dot product == cosine similarity
    embeddings = model.encode(texts, normalize_embeddings=True)

    # build a PointStruct for each chunk: id + vector + payload (metadata)
    points = [
        PointStruct(
            id=i,                           # simple integer ID -- fine for a demo
            vector=embeddings[i].tolist(),  # numpy array -> plain Python list for serialization
            payload={
                "text": CHUNKS[i]["text"],      # store original text so search results are self-contained
                "source": CHUNKS[i]["source"],  # which source file this chunk came from
                "doc_type": CHUNKS[i]["doc_type"],  # used for filtering later
            },
        )
        for i in range(len(CHUNKS))
    ]

    # upsert = insert if new, replace if ID already exists
    client.upsert(collection_name=COLLECTION_NAME, points=points)

    # confirm how many points are now in the collection
    info = client.get_collection(COLLECTION_NAME)
    print(f"Upserted {info.points_count} points")


def search_unfiltered(query: str, k: int = 3, ef: int = 128) -> None:
    # embed the query with the same model used at ingest time -- must always match
    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    # query_points replaces the old client.search() which was removed in qdrant-client >= 1.9
    # .points unwraps the result object to get the actual list of scored hits
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,         # the embedded query vector to search against
        limit=k,                    # return the top-k most similar points
        search_params=SearchParams(
            hnsw_ef=ef              # higher ef = more thorough search = better recall, slower
        ),
        with_payload=True,          # include the payload (text, source, doc_type) in results
    ).points                        # .points extracts the list from the response wrapper

    print(f"\nUnfiltered search: '{query}' (ef={ef})")
    for r in results:
        # r.score is the cosine similarity -- closer to 1.0 = more similar
        print(f"  score={r.score:.4f} | {r.payload['text'][:70]}")


def search_filtered(query: str, doc_type: str, k: int = 3) -> None:
    # same embedding step as unfiltered search -- model must always be the same
    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=Filter(
            must=[                          # ALL conditions in must[] must be true
                FieldCondition(
                    key="doc_type",         # the payload field to filter on
                    match=MatchValue(value=doc_type),  # must equal this exact value
                )
            ]
        ),
        limit=k,
        with_payload=True,
    ).points

    print(f"\nFiltered search (doc_type='{doc_type}'): '{query}'")
    for r in results:
        print(f"  score={r.score:.4f} | [{r.payload['doc_type']}] {r.payload['text'][:60]}")


# ----------------------------------------------------------------------
# Run all demos in order
# ----------------------------------------------------------------------

# step 1: create the collection with explicit HNSW settings
create_hnsw_collection(m=16, ef_construction=100)

# step 2: embed and store all sample chunks
upsert_chunks()

# step 3: search across all doc types -- no filter
search_unfiltered("What happens if I close my FD early?", k=3, ef=64)

# step 4: same query but only look inside FAQ documents
search_filtered("What happens if I close my FD early?", doc_type="faq", k=2)

# step 5: different query, only look inside policy documents
search_filtered("penalty for early closure", doc_type="policy", k=3)

# step 6: compare low ef vs high ef on the same query
# at 8 points the scores will be identical -- the difference shows at scale
print("\n--- ef comparison (difference visible at scale, not at 8 points) ---")
search_unfiltered("senior citizen interest rate", k=2, ef=10)   # fast, lower recall at scale
search_unfiltered("senior citizen interest rate", k=2, ef=200)  # slower, higher recall at scale

Created collection with HNSW m=16, ef_construction=100
Upserted 8 points

Unfiltered search: 'What happens if I close my FD early?' (ef=64)
  score=0.5106 | Can I withdraw my FD before maturity? Yes, subject to a penalty.
  score=0.4080 | This penalty does not apply if the FD is closed due to the death of th
  score=0.3879 | The FD interest rate for 24 months is 7.1 percent per annum.

Filtered search (doc_type='faq'): 'What happens if I close my FD early?'
  score=0.5106 | [faq] Can I withdraw my FD before maturity? Yes, subject to a pena
  score=0.3675 | [faq] What is the minimum amount to open an FD? The minimum deposi

Filtered search (doc_type='policy'): 'penalty for early closure'
  score=0.6175 | [policy] Premature withdrawal incurs a 1 percent penalty on the appli
  score=0.4741 | [policy] This penalty does not apply if the FD is closed due to the d
  score=0.0928 | [policy] TDS is deducted at source if interest income exceeds Rs 40,0

--- ef comparison (difference visible at s

This chapter has 10 topics total:

- What a Vector DB Solves That SQL Can't
- Comparison — FAISS vs. Chroma vs. Qdrant vs. Pinecone vs. Weaviate
- Why Qdrant Specifically
- Indexing — HNSW Explained
- Qdrant's Model — Collections, Points, Payloads
- Migrating the In-Memory Store into Qdrant
- Metadata Filtering in Practice
- Embedding Model Migration
- PII and Access-Scoping
- The Honest Migration Trigger — When In-Memory Stops Being Enough